In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(r'데이터\3,4번. 재무비율 생성+라벨 생성.parquet')
print(f"전체: {len(df):,}행  |  유니크 기업: {df['사업자등록번호'].nunique():,}개")

전체: 343,410행  |  유니크 기업: 50,341개


참고로

모델 학습 직전에

X = manufacturing.drop(
    columns=[
        '회사명',
        '사업자등록번호',
        '통계청 한국표준산업분류 코드 11차(대분류)',
        '통계청 한국표준산업분류 코드 11차(중분류)',
        '부실라벨_ICR3년'
    ]
)

y = manufacturing['부실라벨_ICR3년']

이렇게 제거하면 됨. 지금 이것들을 제거하지 않아도 됨.

## 산업 M코드로 매핑해서 나누기

In [12]:
# ── M코드 매핑 (대분류 숫자코드 기준) ─────────────────────────────
code_to_mcode = {
    1: 'M01', 2: 'M01', 3: 'M01',
    5: 'M02', 6: 'M02', 7: 'M02', 8: 'M02',
    10: 'M03', 11: 'M03', 12: '미분류',
    13: 'M04', 14: 'M04', 15: 'M04',
    16: 'M05', 17: 'M05',
    18: 'M06',
    19: 'M07',
    20: 'M08', 21: 'M08', 22: 'M08',
    23: 'M09',
    24: 'M10',
    25: 'M11',
    26: 'M13', 27: 'M14', 28: 'M13',
    29: 'M12',
    30: 'M15', 31: 'M15',
    32: 'M16', 33: 'M16',
    34: 'M12',
    35: 'M17', 36: 'M17', 37: 'M17', 38: 'M17', 39: 'M17',
    41: 'M18', 42: 'M18',
    45: 'M19', 46: 'M19', 47: 'M19',
    49: 'M21', 50: 'M21', 51: 'M21', 52: 'M21',
    55: 'M20', 56: 'M20',
    58: 'M22', 59: 'M22', 60: 'M22', 61: 'M22', 62: 'M22', 63: 'M22',
    64: '미분류', 65: '미분류', 66: '미분류',
    68: 'M23', 70: 'M23', 71: 'M23', 72: 'M23', 73: 'M23',
    74: 'M23', 75: 'M23', 76: 'M23',
    84: '미분류', 85: 'M24', 86: 'M24', 87: 'M24',
    90: 'M25', 91: 'M25',
    94: 'M25', 95: 'M25', 96: 'M25',
    97: '미분류', 98: '미분류', 99: '미분류',
}

m_info = {
    'M01': '농업_임업_어업',
    'M02': '광업',
    'M03': '음식료품_제조업',
    'M04': '섬유_가죽_신발_제조업',
    'M05': '목재_펄프_종이_제조업',
    'M06': '출판_인쇄_기록매체_복제업',
    'M07': '코크스_석유정제품_제조업',
    'M08': '화학_의약품_고무_플라스틱_제조업',
    'M09': '비금속광물제품_제조업',
    'M10': '제1차금속산업',
    'M11': '조립금속제품_제조업',
    'M12': '기타기계장비_제조업',
    'M13': '전자부품_컴퓨터_전기장비_제조업',
    'M14': '의료_정밀_광학기기_제조업',
    'M15': '운송장비_제조업',
    'M16': '기타제품_제조업',
    'M17': '전기_가스_수도사업',
    'M18': '건설업',
    'M19': '도매_소매업',
    'M20': '숙박_음식점업',
    'M21': '운수_창고업',
    'M22': '정보통신업',
    'M23': '부동산_임대_사업서비스업',
    'M24': '교육_보건_사회복지',
    'M25': '오락_문화_개인서비스업',
}

COL_대분류 = next(
    c for c in [
        '통계청 한국표준산업분류 코드 11차(대분류)',
        '통계청 한국표준산업분류 11차(대분류)',
    ]
    if True  # 실제 컬럼 탐지는 로드 후 수행
)
THRESHOLD = 500  # 기업 수 500개 미만 산업 제외

# ── 컬럼 자동 탐지 ────────────────────────────────────────────────
대분류_후보 = [
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 11차(대분류)',
]
col_대분류 = next((c for c in 대분류_후보 if c in df.columns), None)
if col_대분류 is None:
    raise ValueError(f"대분류 컬럼 없음. 실제 컬럼: {df.columns.tolist()}")
print(f"대분류 컬럼: {col_대분류}")

# ── M코드 부여 및 미분류 제거 ─────────────────────────────────────
df['M코드'] = pd.to_numeric(df[col_대분류], errors='coerce').map(code_to_mcode).fillna('미분류')

before = df['사업자등록번호'].nunique()
df = df[df['M코드'] != '미분류'].copy()
after  = df['사업자등록번호'].nunique()
print(f"미분류 제거: {before - after:,}개 기업 제거")

# ── 기업별 M코드 최빈값으로 고정 (복수 산업 걸친 기업 처리) ──────────
mcode_mode = (
    df.groupby('사업자등록번호')['M코드']
    .agg(lambda x: x.mode().iloc[0])
    .rename('M코드_고정')
)
multi = mcode_mode.index[
    df.groupby('사업자등록번호')['M코드'].nunique() > 1
]
if len(multi) > 0:
    print(f"\n2개 이상 산업에 걸친 기업 {len(multi):,}개 → 최빈 M코드로 통일:")
    for biz in multi:
        orig = df.loc[df['사업자등록번호'] == biz, 'M코드'].value_counts().to_dict()
        print(f"  {biz}: {orig} → {mcode_mode[biz]}")

df['M코드'] = df['사업자등록번호'].map(mcode_mode)
print(f"고정 후 유니크 기업: {df['사업자등록번호'].nunique():,}개")

# ── 기업 수 500개 미만 산업 제외 ──────────────────────────────────
company_count = df.groupby('M코드')['사업자등록번호'].nunique()
valid_mcodes  = company_count[company_count >= THRESHOLD].index
excl_mcodes   = company_count[company_count <  THRESHOLD].index

if len(excl_mcodes) > 0:
    print(f"\n기업 수 {THRESHOLD}개 미만 제외 산업:")
    for m in sorted(excl_mcodes):
        print(f"  {m} {m_info.get(m, m)}: {company_count[m]:,}개")

df = df[df['M코드'].isin(valid_mcodes)].copy()
print(f"\n저장 대상: {df['사업자등록번호'].nunique():,}개 기업  |  {len(valid_mcodes)}개 산업\n")


대분류 컬럼: 통계청 한국표준산업분류 코드 11차(대분류)
미분류 제거: 388개 기업 제거

2개 이상 산업에 걸친 기업 1개 → 최빈 M코드로 통일:
  6218161477: {'M08': 4, 'M13': 1} → M08
고정 후 유니크 기업: 49,953개

기업 수 500개 미만 제외 산업:
  M01 농업_임업_어업: 266개
  M02 광업: 124개
  M05 목재_펄프_종이_제조업: 489개
  M06 출판_인쇄_기록매체_복제업: 212개
  M07 코크스_석유정제품_제조업: 70개
  M16 기타제품_제조업: 423개
  M24 교육_보건_사회복지: 221개

저장 대상: 48,148개 기업  |  18개 산업



### 저장

In [13]:
# ── 출력 폴더 생성 ────────────────────────────────────────────────
out_dir = Path(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\5번. 산업별 데이터')
out_dir.mkdir(exist_ok=True)

# ── 산업별 parquet 저장 ─────────────────────────────────────
print(f"{'M코드':<6}  {'산업명':<35}  {'행 수':>8}  {'기업 수':>8}")
print("-" * 68)

for mcode in sorted(df['M코드'].unique()):
    subset = df[df['M코드'] == mcode].copy()
    name   = m_info.get(mcode, mcode)
    stem   = out_dir / f"{mcode}_{name}"

    subset.to_parquet(f"{stem}.parquet", index=False, compression='snappy')

    print(f"{mcode:<6}  {name:<35}  {len(subset):>8,}  {subset['사업자등록번호'].nunique():>8,}")

print("-" * 68)
print(f"\n저장 완료: '{out_dir}/' 폴더에 {df['M코드'].nunique()}개 산업 × parquet")


M코드     산업명                                       행 수      기업 수
--------------------------------------------------------------------
M03     음식료품_제조업                                9,787     1,307
M04     섬유_가죽_신발_제조업                            9,397     1,214
M08     화학_의약품_고무_플라스틱_제조업                     24,668     2,963
M09     비금속광물제품_제조업                             6,400       777
M10     제1차금속산업                                 9,689     1,131
M11     조립금속제품_제조업                             13,019     1,663
M12     기타기계장비_제조업                             21,140     2,614
M13     전자부품_컴퓨터_전기장비_제조업                      22,492     2,908
M14     의료_정밀_광학기기_제조업                          4,738       639
M15     운송장비_제조업                               17,647     2,080
M17     전기_가스_수도사업                              6,867     1,021
M18     건설업                                    29,392     4,463
M19     도매_소매업                                 43,966     6,124
M20     숙박_음식점업                    